In [1]:
# Local rootpath
rootpath = "../Data/"

In [2]:
!pip install tensorflow-recommenders

  Using cached tensorflow_recommenders-0.7.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached h5py-3.14.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-4.25.8-cp37-abi3-macosx_10_9_universal2.whl.metadata (541 bytes)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.74.0-cp311-cp311-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached keras-3.11.3-py3-none

In [3]:
import pandas as pd

In [4]:
events = pd.read_csv(rootpath + 'events.csv')
events

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN
...,...,...,...,...,...
2756096,1438398785939,591435,view,261427,NaN
2756097,1438399813142,762376,view,115946,NaN
2756098,1438397820527,1251746,view,78144,NaN
2756099,1438398530703,1184451,view,283392,NaN


In [5]:
events["event"].unique()

array(['view', 'addtocart', 'transaction'], dtype=object)

In [7]:
# Group by visitorid and itemid, aggregate all event types into a set
grouped = events.groupby(['visitorid', 'itemid']).agg({'event': lambda x: set(x)})

# Add boolean indicator columns for each event type
grouped['is_view'] = grouped['event'].apply(lambda x: 'view' in x)
grouped['is_transaction'] = grouped['event'].apply(lambda x: 'transaction' in x)
grouped['is_addtocart'] = grouped['event'].apply(lambda x: 'addtocart' in x)

In [8]:
grouped

event  is_view  is_transaction  is_addtocart
visitorid itemid                                               
0         67045   {view}     True           False         False
          285930  {view}     True           False         False
          357564  {view}     True           False         False
1         72028   {view}     True           False         False
2         216305  {view}     True           False         False
...                  ...      ...             ...           ...
1407575   121220  {view}     True           False         False
1407576   356208  {view}     True           False         False
1407577   427784  {view}     True           False         False
1407578   188736  {view}     True           False         False
1407579   2521    {view}     True           False         False

[2145179 rows x 4 columns]

In [9]:
# Create a pivot table: counts of each event type per (visitorid, itemid)
pivot_df = pd.pivot_table(
    events,
    index=['visitorid', 'itemid'],
    columns='event',
    aggfunc='size',
    fill_value=0
)

# Add indicator columns for each event type
pivot_df['is_view'] = (pivot_df['view'] > 0).astype(int)
pivot_df['is_transaction'] = (pivot_df['transaction'] > 0).astype(int)
pivot_df['is_addtocart'] = (pivot_df['addtocart'] > 0).astype(int)

In [10]:
pivot_df

event             addtocart  transaction  view  is_view  is_transaction  \
visitorid itemid                                                          
0         67045           0            0     1        1               0   
          285930          0            0     1        1               0   
          357564          0            0     1        1               0   
1         72028           0            0     1        1               0   
2         216305          0            0     2        1               0   
...                     ...          ...   ...      ...             ...   
1407575   121220          0            0     1        1               0   
1407576   356208          0            0     1        1               0   
1407577   427784          0            0     1        1               0   
1407578   188736          0            0     1        1               0   
1407579   2521            0            0     1        1               0   

event             is_addtocart  
visitorid itemid                
0         67045              0  
          285930             0  
          357564             0  
1         72028              0  
2         216305             0  
...                        ...  
1407575   121220             0  
1407576   356208             0  
1407577   427784             0  
1407578   188736             0  
1407579   2521               0  

[2145179 rows x 6 columns]

In [11]:
pivot_df.count()

event
addtocart         2145179
transaction       2145179
view              2145179
is_view           2145179
is_transaction    2145179
is_addtocart      2145179
dtype: int64

In [12]:
pivot_df[pivot_df.is_view + pivot_df.is_transaction + pivot_df.is_addtocart > 1].count()

event
addtocart         52890
transaction       52890
view              52890
is_view           52890
is_transaction    52890
is_addtocart      52890
dtype: int64

In [13]:
pivot_df[pivot_df.is_view + pivot_df.is_transaction + pivot_df.is_addtocart > 1]

event             addtocart  transaction  view  is_view  is_transaction  \
visitorid itemid                                                          
150       452955          1            0     2        1               0   
172       10034           1            1     5        1               1   
          465522          2            1     3        1               1   
177       6073            1            0     2        1               0   
186       49029           1            1     2        1               1   
...                     ...          ...   ...      ...             ...   
1407437   389814          1            0     2        1               0   
1407512   326819          1            0     1        1               0   
          334964          1            0     1        1               0   
          398091          1            0     2        1               0   
          398533          1            0     1        1               0   

event             is_addtocart  
visitorid itemid                
150       452955             1  
172       10034              1  
          465522             1  
177       6073               1  
186       49029              1  
...                        ...  
1407437   389814             1  
1407512   326819             1  
          334964             1  
          398091             1  
          398533             1  

[52890 rows x 6 columns]

## Predict click, add-to-cart, and purchase counts

In [14]:
df = pivot_df.reset_index()

In [15]:
df

event,visitorid,itemid,addtocart,transaction,view,is_view,is_transaction,is_addtocart
0,0,67045,0,0,1,1,0,0
1,0,285930,0,0,1,1,0,0
2,0,357564,0,0,1,1,0,0
3,1,72028,0,0,1,1,0,0
4,2,216305,0,0,2,1,0,0
...,...,...,...,...,...,...,...,...
2145174,1407575,121220,0,0,1,1,0,0
2145175,1407576,356208,0,0,1,1,0,0
2145176,1407577,427784,0,0,1,1,0,0
2145177,1407578,188736,0,0,1,1,0,0


In [ ]:
df = df[df.view + df.addtocart + df.transaction > 1] 

In [17]:
df

event,visitorid,itemid,addtocart,transaction,view,is_view,is_transaction,is_addtocart
4,2,216305,0,0,2,1,0,0
6,2,325215,0,0,3,1,0,0
7,2,342816,0,0,2,1,0,0
13,6,344723,0,0,4,1,0,0
32,22,86411,0,0,2,1,0,0
...,...,...,...,...,...,...,...,...
2145121,1407530,455539,0,0,2,1,0,0
2145128,1407537,156770,0,0,2,1,0,0
2145151,1407560,240708,0,0,2,1,0,0
2145164,1407572,122041,0,0,2,1,0,0


In [18]:
import tensorflow as tf

In [19]:
FEATURES = ['visitorid', 'itemid']
LABELS = ['addtocart', 'transaction', 'view']

In [20]:
ds = tf.data.Dataset.from_tensor_slices(dict(df[FEATURES + LABELS]))

In [21]:
ds_cache = ds.map(lambda x: (
            { feature: x[feature] for feature in FEATURES },
            { label: x[label] for label in LABELS })) \
    .shuffle(10000) \
    .batch(500, drop_remainder=True) \
    .cache()

In [22]:
ds_cache.element_spec

({'visitorid': TensorSpec(shape=(500,), dtype=tf.int64, name=None),
  'itemid': TensorSpec(shape=(500,), dtype=tf.int64, name=None)},
 {'addtocart': TensorSpec(shape=(500,), dtype=tf.int64, name=None),
  'transaction': TensorSpec(shape=(500,), dtype=tf.int64, name=None),
  'view': TensorSpec(shape=(500,), dtype=tf.int64, name=None)})

In [23]:
import tensorflow_recommenders as tfrs

In [24]:
def create_model():
    # Input layers for visitorid and itemid
    visitorid = tf.keras.Input(shape=(), dtype='int64', name='visitorid')
    itemid = tf.keras.Input(shape=(), dtype='int64', name='itemid')
    
    # Embedding pipeline for visitorid: Hashing + Embedding
    embedding1 = tf.keras.Sequential([
        tf.keras.layers.Hashing(1e6),                        # Hash visitorid to fixed-size integer space
        tf.keras.layers.Embedding(int(1e6 + 1), 8)           # Learnable embedding vector of size 8
    ])
    visitor_embed = embedding1(visitorid)
    
    # Embedding pipeline for itemid: Hashing + Embedding
    embedding2 = tf.keras.Sequential([
        tf.keras.layers.Hashing(1e6),                        # Hash itemid to fixed-size integer space
        tf.keras.layers.Embedding(int(1e6 + 1), 8)           # Learnable embedding vector of size 8
    ])
    item_embed = embedding2(itemid)
    
    # Concatenate visitor and item embeddings
    concat_input = tf.keras.layers.Concatenate(axis=-1)([visitor_embed, item_embed])
    
    # Hidden layers with dropout for regularization
    x = tf.keras.layers.Dense(16, activation='relu')(concat_input)
    x = tf.keras.layers.Dropout(0.1)(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    
    # Separate output layers for multitask predictions
    view_pred = tf.keras.layers.Dense(1, name='view')(x)            # Predict click (view)
    addtocart_pred = tf.keras.layers.Dense(1, name='addtocart')(x)  # Predict add-to-cart
    transaction_pred = tf.keras.layers.Dense(1, name='transaction')(x)  # Predict purchase
    
    # Build and return the multitask model
    return tf.keras.Model(inputs=[visitorid, itemid], 
                          outputs=[view_pred, addtocart_pred, transaction_pred])

In [25]:
model = create_model()

In [26]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ visitorid           │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ itemid (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 8)         │  8,000,008 │ visitorid[0][0]   │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 8)         │  8,000,008 │ itemid[0][0]      │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16)        │          0 │ sequential[0][0], │
│ (Concatenate)       │                   │            │ sequential_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        272 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        272 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 16)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ view (Dense)        │ (None, 1)         │         17 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ addtocart (Dense)   │ (None, 1)         │         17 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transaction (Dense) │ (None, 1)         │         17 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,000,611 (61.04 MB)

 Trainable params: 16,000,611 (61.04 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
feature, label = next(iter(ds_cache))

2025-09-03 22:38:04.528855: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [32]:
model(feature)

[<tf.Tensor: shape=(500, 1), dtype=float32, numpy=
 array([[ 2.87014735e-03],
        [ 2.26954054e-02],
        [ 4.22124378e-03],
        [-4.26595332e-03],
        [ 8.38261284e-03],
        [-8.29517841e-04],
        [ 8.13277997e-03],
        [ 2.08134670e-02],
        [ 1.47612579e-02],
        [ 2.24485155e-03],
        [ 3.34650688e-02],
        [-1.04761366e-02],
        [ 1.44583937e-02],
        [ 1.36004128e-02],
        [ 5.09932917e-03],
        [ 6.78926765e-04],
        [ 3.40856239e-03],
        [ 2.90766824e-04],
        [-8.76945350e-03],
        [ 3.14864218e-02],
        [-4.56257816e-03],
        [ 7.00395647e-03],
        [-8.02381663e-04],
        [ 9.07251425e-03],
        [ 2.60393019e-03],
        [ 4.11769152e-02],
        [ 1.31027084e-02],
        [ 3.43844201e-03],
        [ 8.09547637e-05],
        [-1.11776125e-02],
        [ 1.91293098e-03],
        [-4.03360697e-03],
        [ 5.02994843e-03],
        [-4.32796776e-04],
        [ 4.13111411e-05],
    

In [33]:
model.compile(optimizer='adam',
              loss={
                    'view': 'mse',
                    'addtocart': 'mse',
                    'transaction': 'mse'
                },
              metrics=['accuracy'])

In [42]:
# 1) Build tf.data.Dataset that matches model I/O names
df = pivot_df.reset_index()  # ensure visitorid, itemid are columns

# Features dict must match model input names
X = {
    "visitorid": df["visitorid"].astype("int64").values,
    "itemid": df["itemid"].astype("int64").values,
}

# Labels dict must match model output names
Y = {
    "view": df["is_view"].astype("float32").values,
    "addtocart": df["is_addtocart"].astype("float32").values,
    "transaction": df["is_transaction"].astype("float32").values,
}

import tensorflow as tf
ds_cache = (
    tf.data.Dataset.from_tensor_slices((X, Y))
    .batch(1024)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# 2) Compile (choose ONE of the options below)
# Option A: if your output layers use sigmoid activation
model.compile(
    optimizer="adam",
    loss={
        "view": "binary_crossentropy",
        "addtocart": "binary_crossentropy",
        "transaction": "binary_crossentropy",
    },
    metrics={"view": ["accuracy"], "addtocart": ["accuracy"], "transaction": ["accuracy"]},
)

# Option B: if your output layers have NO activation (logits)
# from tensorflow.keras.losses import BinaryCrossentropy
# model.compile(
#     optimizer="adam",
#     loss={
#         "view": BinaryCrossentropy(from_logits=True),
#         "addtocart": BinaryCrossentropy(from_logits=True),
#         "transaction": BinaryCrossentropy(from_logits=True),
#     },
#     metrics={"view": ["accuracy"], "addtocart": ["accuracy"], "transaction": ["accuracy"]},
# )

# 3) Fit
model.fit(ds_cache, epochs=3)


Epoch 1/3
2095/2095 ━━━━━━━━━━━━━━━━━━━━ 61s 28ms/step - addtocart_accuracy: 0.9711 - addtocart_loss: 0.2303 - loss: 0.6356 - transaction_accuracy: 0.9901 - transaction_loss: 0.1393 - view_accuracy: 0.9047 - view_loss: 0.2660
Epoch 2/3
2095/2095 ━━━━━━━━━━━━━━━━━━━━ 74s 35ms/step - addtocart_accuracy: 0.9711 - addtocart_loss: 0.1766 - loss: 0.3670 - transaction_accuracy: 0.9900 - transaction_loss: 0.1029 - view_accuracy: 0.9924 - view_loss: 0.0875
Epoch 3/3
2095/2095 ━━━━━━━━━━━━━━━━━━━━ 81s 39ms/step - addtocart_accuracy: 0.9711 - addtocart_loss: 0.1346 - loss: 0.2545 - transaction_accuracy: 0.9901 - transaction_loss: 0.0647 - view_accuracy: 0.9930 - view_loss: 0.0552


In [44]:
model.save(rootpath + 'multitask.keras')